# Fase 3 — Notebook 4: Entrenamiento Final y Evaluación en Test

**Objetivo:** Entrenar el modelo ganador usando el conjunto de train completo
(datos reales + datos sintéticos del VAE) y evaluarlo **una única vez**
sobre el 20% de test que nunca fue tocado.

**Flujo:**
```
Train real (clase 0 + clase 1)  ┐
                                 ├─→ Modelo final ─→ Evaluación en TEST (una vez)
Sintéticos VAE (clase 1)        ┘
```

**Threshold:** calibrado sobre un 15% interno del train (estratificado, antes
de mezclar con sintéticos) — igual que en las fases anteriores.

**Métricas reportadas:** Accuracy, Precision, Recall, F1 (pos), F1-Macro, AUC-ROC.

**Comparación final:** los resultados se contrastan contra la mini CV del
notebook 2 para verificar que el augmentation ayudó.

---
**Prerequisitos:** notebooks 1, 2 y 3 completados.

In [43]:
# ─── Celda 1: Imports y configuración ─────────────────────────────────────
import re
import gc
import json
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision.transforms as T
from transformers import ASTConfig, ASTModel

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, auc
)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════════════
CONDITION       = "Ansiedad"
SPECS_ROOT      = Path("Clasificación Final/Ansiedad/specs")
SPLITS_DIR      = Path("Ansiedad/fase3/splits")
SINTETICOS_ROOT = Path("Ansiedad/fase3/sinteticos")
RESULTS_ROOT    = Path("Ansiedad/fase3/resultado_final")
MINI_CV_ROOT    = Path("Ansiedad/fase3/mini_cv")

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

VAL_FRAC  = 0.15
THR_GRID  = np.arange(0.05, 0.96, 0.02)

# Hiperparametros por tipo de modelo
HP = {
    "CNN2D": {"batch": 128, "epochs": 30, "patience": 5,  "lr": 1e-4},
    "AST":   {"batch": 32,  "epochs": 20, "patience": 4,  "lr": 2e-5,
              "n_unfreeze": 3},
}
AST_PRETRAINED   = "MIT/ast-finetuned-audioset-10-10-0.4593"
OLD_FREQ_PATCHES = 12
OLD_TIME_PATCHES = 101

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# En lugar de cargar solo al ganador, cargamos la lista completa para entrenarlos a todos.
todos_path = MINI_CV_ROOT / "todos_candidatos.json"
if todos_path.exists():
    with open(todos_path) as f:
        CANDIDATOS = json.load(f)["candidatos"]
    print(f"\n[CARGADO] Lista de {len(CANDIDATOS)} candidatos desde {todos_path}")

print(f"Candidatos a procesar: {[c['id'] for c in CANDIDATOS]}")

Dispositivo: cuda
  GPU: NVIDIA GeForce RTX 5080

[CARGADO] Lista de 4 candidatos desde Ansiedad/fase3/mini_cv/todos_candidatos.json
Candidatos a procesar: ['CNN2D_logmel_2s', 'AST_gammatone_2s', 'AST_gammatone_3s', 'AST_gammatone_4s']


In [44]:
# ─── Celda 2: Carga de splits y construcción de índices ───────────────────

def load_full_split(filter_name, duration):
    json_path = SPLITS_DIR / f"split_{filter_name}_{duration}s.json"
    with open(json_path) as f:
        sp = json.load(f)
    train_stems  = sp["train"]["stems"]
    train_labels = np.array(sp["train"]["labels"], dtype=np.int64)
    test_stems   = sp["test"]["stems"]
    test_labels  = np.array(sp["test"]["labels"],  dtype=np.int64)
    print(f"Train: {len(train_stems)} pac "
          f"(0:{sp['train']['n_class0']} / 1:{sp['train']['n_class1']})")
    print(f"Test : {len(test_stems)} pac "
          f"(0:{sp['test']['n_class0']}  / 1:{sp['test']['n_class1']})")
    return train_stems, train_labels, test_stems, test_labels


def build_img_map(root: Path):
    img_map = {}
    for cls_str in ["0", "1"]:
        d = root / cls_str
        if not d.exists(): continue
        for p in sorted(d.glob("*.png")):
            stem = re.sub(r'_seg\d+$', '', p.stem)
            img_map.setdefault(stem, []).append(p)
    return img_map


def get_target_width(img_map, stems):
    return max(Image.open(img_map[s][0]).size[0]
               for s in stems if s in img_map)

for cand in CANDIDATOS:
    # 1️ Definir variables por candidato (reemplaza las globales que quitamos en C1)
    cid         = cand["id"]
    MODEL_TYPE  = cand["model"]
    FILTER_NAME = cand["filter"]
    DURATION    = cand["dur"]
    IMG_H       = cand["img_h"]
    AST_H       = cand.get("ast_h", IMG_H)

    print(f"\n{'='*60}")
    print(f"  CANDIDATO: {cid}")
    print(f"  Tipo: {MODEL_TYPE} | Filtro: {FILTER_NAME} | Dur: {DURATION}s")
    print(f"{'='*60}")

    train_stems, train_labels, test_stems, test_labels = load_full_split(
        FILTER_NAME, DURATION)

    image_root = SPECS_ROOT / FILTER_NAME / f"{DURATION}s"
    img_map    = build_img_map(image_root)

    sintetico_dir = SINTETICOS_ROOT / FILTER_NAME / f"{DURATION}s" / "1_sintetico"
    sintetico_paths = sorted(sintetico_dir.glob("*.png")) if sintetico_dir.exists() else []
    print(f"\nPNGs sinteticos disponibles: {len(sintetico_paths):,}")

    if MODEL_TYPE == "AST":
        target_w = get_target_width(img_map, train_stems)
        print(f"AST target_w: {target_w}px")
    else:
        target_w = None

    # Limpieza ligera antes del siguiente candidato
    gc.collect()


  CANDIDATO: CNN2D_logmel_2s
  Tipo: CNN2D | Filtro: log-mel | Dur: 2s
Train: 63 pac (0:48 / 1:15)
Test : 16 pac (0:12  / 1:4)

PNGs sinteticos disponibles: 4,882

  CANDIDATO: AST_gammatone_2s
  Tipo: AST | Filtro: gammatone | Dur: 2s
Train: 63 pac (0:48 / 1:15)
Test : 16 pac (0:12  / 1:4)

PNGs sinteticos disponibles: 4,882
AST target_w: 63px

  CANDIDATO: AST_gammatone_3s
  Tipo: AST | Filtro: gammatone | Dur: 3s
Train: 63 pac (0:48 / 1:15)
Test : 16 pac (0:12  / 1:4)

PNGs sinteticos disponibles: 3,243
AST target_w: 94px

  CANDIDATO: AST_gammatone_4s
  Tipo: AST | Filtro: gammatone | Dur: 4s
Train: 63 pac (0:48 / 1:15)
Test : 16 pac (0:12  / 1:4)

PNGs sinteticos disponibles: 2,421
AST target_w: 126px


In [52]:
# ─── Celda 3: Datasets ────────────────────────────────────────────────────

# ── Transforms ────────────────────────────────────────────────────────────
CNN_TF_TRAIN = T.Compose([
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
CNN_TF_VAL = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
AST_TF = T.Compose([
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5]),
])


# ── Dataset para datos REALES (CNN y AST) ─────────────────────────────────
class RealDataset(Dataset):
    def __init__(self, stems, img_map, transform, mode="CNN",
                 ast_h=None, target_w=None):
        self.samples  = []
        self.transform = transform
        self.mode     = mode
        self.ast_h    = ast_h
        self.target_w = target_w
        for stem in stems:
            label = int(img_map[stem][0].parent.name)
            for p in img_map[stem]:
                self.samples.append((p, label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.mode == "CNN":
            return self.transform(img), torch.tensor(label, dtype=torch.long)
        else:  # AST
            t = self.transform(img).squeeze(0)
            if t.shape[0] < self.ast_h:
                t = F.pad(t, (0, 0, 0, self.ast_h - t.shape[0]), value=-1.0)
            if t.shape[1] < self.target_w:
                t = F.pad(t, (0, self.target_w - t.shape[1]), value=-1.0)
            return t.T, torch.tensor(label, dtype=torch.long)


# ── Dataset para datos SINTÉTICOS (siempre clase 1) ───────────────────────
class SinteticoDataset(Dataset):
    def __init__(self, paths, transform, mode="CNN",
                 ast_h=None, target_w=None):
        self.paths    = paths
        self.transform = transform
        self.mode     = mode
        self.ast_h    = ast_h
        self.target_w = target_w

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.mode == "CNN":
            return self.transform(img), torch.tensor(1, dtype=torch.long)
        else:  # AST
            t = self.transform(img).squeeze(0)
            if t.shape[0] < self.ast_h:
                t = F.pad(t, (0, 0, 0, self.ast_h - t.shape[0]), value=-1.0)
            if t.shape[1] < self.target_w:
                t = F.pad(t, (0, self.target_w - t.shape[1]), value=-1.0)
            return t.T, torch.tensor(1, dtype=torch.long)


# ── Dataset para INFERENCIA por stem (segment voting) ─────────────────────
class StemDataset(Dataset):
    def __init__(self, stems, img_map, transform, mode="CNN",
                 ast_h=None, target_w=None):
        self.samples   = [(p, s) for s in stems for p in img_map[s]]
        self.transform = transform
        self.is_cnn = (mode=="CNN" or mode == "CN2D")
        self.mode      = mode
        self.ast_h     = ast_h
        self.target_w  = target_w

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, stem = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.is_cnn:
            return self.transform(img), stem
        else:  # AST
            t = self.transform(img).squeeze(0)
            # Solo aplicar padding si ast_h y target_w no son None
            if self.ast_h is not None and t.shape[0] < self.ast_h:
                t = F.pad(t, (0, 0, 0, self.ast_h - t.shape[0]), value=-1.0)
            if self.target_w is not None and t.shape[1] < self.target_w:
                t = F.pad(t, (0, self.target_w - t.shape[1]), value=-1.0)
            return t.mT, stem # o t.permute(1,0)

print("Datasets definidos")

Datasets definidos


In [46]:
# ─── Celda 4: Modelos ──────────────────────────────────────────────────────

class CNN2D(nn.Module):
    def __init__(self):
        super().__init__()
        def cb(ic, oc, n):
            lys = []
            for i in range(n):
                lys += [nn.Conv2d(ic if i==0 else oc, oc, 3, padding=1),
                        nn.BatchNorm2d(oc), nn.ReLU(inplace=True)]
            return nn.Sequential(*lys, nn.MaxPool2d(2), nn.Dropout2d(0.2))
        self.features = nn.Sequential(
            cb(3,32,2), cb(32,64,2), cb(64,128,3), cb(128,256,3), cb(256,256,3))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256,128), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(128, 64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64,   1), nn.Sigmoid())
    def forward(self, x):
        return self.classifier(self.pool(self.features(x))).squeeze(1)


class ASTClassifier(nn.Module):
    def __init__(self, ast_h, target_w, n_unfreeze=3):
        super().__init__()
        fp = (ast_h    - 16) // 10 + 1
        tp = (target_w - 16) // 10 + 1
        config = ASTConfig(
            num_mel_bins=ast_h, max_length=target_w,
            hidden_size=768, num_hidden_layers=12,
            num_attention_heads=12, intermediate_size=3072,
            patch_size=16, frequency_stride=10, time_stride=10)
        pretrained = ASTModel.from_pretrained(AST_PRETRAINED)
        self.ast   = ASTModel(config)
        self.ast.encoder.load_state_dict(pretrained.encoder.state_dict())
        self.ast.layernorm.load_state_dict(pretrained.layernorm.state_dict())
        self.ast.embeddings.patch_embeddings.load_state_dict(
            pretrained.embeddings.patch_embeddings.state_dict())
        self.ast.embeddings.cls_token.data = \
            pretrained.embeddings.cls_token.data.clone()
        self.ast.embeddings.distillation_token.data = \
            pretrained.embeddings.distillation_token.data.clone()
        old = pretrained.embeddings.position_embeddings.data
        sp  = old[:,:2,:]; pt = old[:,2:,:]; h = pt.shape[-1]
        pt  = pt.reshape(1,OLD_FREQ_PATCHES,OLD_TIME_PATCHES,h).permute(0,3,1,2)
        pt  = F.interpolate(pt.float(), size=(fp,tp), mode='bilinear', align_corners=False)
        pt  = pt.permute(0,2,3,1).reshape(1,fp*tp,h)
        self.ast.embeddings.position_embeddings = nn.Parameter(torch.cat([sp,pt],1))
        del pretrained; gc.collect()
        self.classifier = nn.Sequential(
            nn.Linear(768,256), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(256,1),   nn.Sigmoid())
        for p in self.parameters(): p.requires_grad = False
        for layer in self.ast.encoder.layer[-n_unfreeze:]:
            for p in layer.parameters(): p.requires_grad = True
        for p in self.ast.layernorm.parameters(): p.requires_grad = True
        for p in self.classifier.parameters():   p.requires_grad = True
        n_tr = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  AST entrenables: {n_tr:,}")
    def forward(self, x):
        return self.classifier(
            self.ast(input_values=x).last_hidden_state[:,0,:]).squeeze(1)

print("Modelos definidos")

Modelos definidos


In [47]:
# ─── Celda 5: Funciones de entrenamiento e inferencia ─────────────────────

def train_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        loss = (criterion(model(xb).view(-1), yb) * cw_tensor[yb.long()]).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


def predict_stems(model, stems, img_map, batch_size, mode,
                  ast_h=None, target_w=None):
    model.eval()
     # ✅ FIX: Detectar correctamente "CNN" o "CNN2D"
    is_cnn = (mode == "CNN" or mode == "CNN2D")
    tf = CNN_TF_VAL if is_cnn else AST_TF

    ds = StemDataset(stems, img_map, tf, mode=mode,
                     ast_h=ast_h, target_w=target_w)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False,
                    num_workers=4, pin_memory=True, persistent_workers=False)

    probs = {s: [] for s in stems}
    with torch.no_grad():
        for xb, snames in dl:
            # ✅ Asegurar formato (B, C, H, W) para CNN
            if is_cnn and xb.ndim == 4:
                if xb.shape[1] != 3:
                    if xb.shape[-1] == 3:          # Caso (B, H, W, C)
                        xb = xb.permute(0, 3, 1, 2)
                    elif xb.shape[3] == 3:         # Caso (B, H, W, C)
                        xb = xb.permute(0, 3, 1, 2)

            out = model(xb.to(DEVICE, non_blocking=True)).view(-1).cpu().numpy()
            for p, s in zip(out, snames): probs[s].append(float(p))
    return {s: float(np.mean(v)) for s, v in probs.items()}


def calibrate_thr(probs_dict, stems, labels):
    arr = np.array([probs_dict[s] for s in stems])
    best_thr, best_f1 = 0.5, -1.0
    for thr in THR_GRID:
        sc = f1_score(labels, (arr > thr).astype(int),
                      average='macro', zero_division=0)
        if sc > best_f1: best_f1, best_thr = sc, float(thr)
    return best_thr

print("Funciones definidas")

Funciones definidas


In [48]:
# ─── Celda 6: Separar val interna del train para calibrar threshold ────────
#
# El 15% de val se separa de los datos REALES antes de añadir sintéticos.
# Esto garantiza que el threshold se calibra con datos limpios.

for cand in CANDIDATOS:
    # 1️ Variables por candidato (reemplaza las globales de Celda 1)
    cid         = cand["id"]
    MODEL_TYPE  = cand["model"]
    FILTER_NAME = cand["filter"]
    DURATION    = cand["dur"]
    IMG_H       = cand["img_h"]
    AST_H       = cand.get("ast_h", IMG_H)

    print(f"\n{'='*60}")
    print(f"  CANDIDATO: {cid}")
    print(f"  Tipo: {MODEL_TYPE} | Filtro: {FILTER_NAME} | Dur: {DURATION}s")
    print(f"{'='*60}")

    # 2️ Cargar split y sintéticos ESPECÍFICOS de este candidato
    json_path = SPLITS_DIR / f"split_{FILTER_NAME}_{DURATION}s.json"
    with open(json_path) as f: sp = json.load(f)
    train_stems  = sp["train"]["stems"]
    train_labels = np.array(sp["train"]["labels"], dtype=np.int64)
    test_stems   = sp["test"]["stems"]
    test_labels  = np.array(sp["test"]["labels"],  dtype=np.int64)

    # ✅ Ruta dinámica de sintéticos (por candidato)
    sintetico_dir = SINTETICOS_ROOT / FILTER_NAME / f"{DURATION}s" / "1_sintetico"
    sintetico_paths = sorted(sintetico_dir.glob("*.png")) if sintetico_dir.exists() else []

    # 3️⃣ Split 85/15 para threshold (con datos REALES antes de augment)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=42)
    tr_sub_idx, val_idx = next(sss.split(train_stems, train_labels))
    val_stems       = [train_stems[i] for i in val_idx]
    val_labels_arr  = train_labels[val_idx]
    real_train_stems = [train_stems[i] for i in tr_sub_idx]
    real_train_lbl   = train_labels[tr_sub_idx]

    # ──────────────────────────────────────────────────────────────────────
    # TUS PRINTS ORIGINALES (INTACTOS)
    # ──────────────────────────────────────────────────────────────────────
    print(f"Val interna (calibracion threshold) : {len(val_stems)} pacientes")
    print(f"  Clase 0: {int((val_labels_arr==0).sum())}")
    print(f"  Clase 1: {int((val_labels_arr==1).sum())}")
    print(f"Train real (sin val)                : {len(real_train_stems)} pacientes")
    print(f"  Clase 0: {int((real_train_lbl==0).sum())}")
    print(f"  Clase 1: {int((real_train_lbl==1).sum())}")
    print(f"Sinteticos clase 1                  : {len(sintetico_paths):,} PNGs")

    # 4️⃣ Construir img_map y target_w por candidato
    image_root = SPECS_ROOT / FILTER_NAME / f"{DURATION}s"
    img_map = {}
    for cls_str in ["0", "1"]:
        d = image_root / cls_str
        if not d.exists(): continue
        for p in sorted(d.glob("*.png")):
            stem = re.sub(r'_seg\d+$', '', p.stem)
            img_map.setdefault(stem, []).append(p)

    target_w = None
    if MODEL_TYPE == "AST":
        valid = [s for s in real_train_stems if s in img_map and len(img_map[s]) > 0]
        if valid: target_w = max(Image.open(img_map[s][0]).size[0] for s in valid)

    # 5️⃣ Datasets real y sintético
    if MODEL_TYPE == "CNN2D":
        ds_real = RealDataset(real_train_stems, img_map, CNN_TF_TRAIN, mode="CNN")
        ds_sint = SinteticoDataset(sintetico_paths, CNN_TF_TRAIN, mode="CNN")
    else:
        ds_real = RealDataset(real_train_stems, img_map, AST_TF, mode="AST",
                              ast_h=AST_H, target_w=target_w)
        ds_sint = SinteticoDataset(sintetico_paths, AST_TF, mode="AST",
                                   ast_h=AST_H, target_w=target_w)

    # Concatenar real + sintético
    ds_train = ConcatDataset([ds_real, ds_sint])
    hp = HP[MODEL_TYPE]
    train_dl = DataLoader(ds_train, batch_size=hp["batch"], shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=False)

    # ──────────────────────────────────────────────────────────────────────
    # ✅ FIX CRÍTICO: class_weight a nivel de SEGMENTO (no paciente)
    # ──────────────────────────────────────────────────────────────────────
    # Contar segmentos reales por clase
    c0_segs = sum(1 for _, l in ds_real if l == 0)
    c1_segs = sum(1 for _, l in ds_real if l == 1)
    # Añadir sintéticos (todos son clase 1)
    c1_segs += len(ds_sint)
    total = c0_segs + c1_segs
    # Fórmula balanced: weight_c = total / (2 * n_c)
    cw_tensor = torch.tensor([total/(2*c0_segs), total/(2*c1_segs)], dtype=torch.float32).to(DEVICE)

    print(f"\n  Class weights (segment level): C0={c0_segs:,}, C1={c1_segs:,} → cw={cw_tensor.tolist()}")

    # ──────────────────────────────────────────────────────────────────────
    # TUS PRINTS ORIGINALES DE RESUMEN (INTACTOS)
    # ──────────────────────────────────────────────────────────────────────
    print(f"\nDataset train aumentado:")
    print(f"  Real    : {len(ds_real):,} segmentos")
    print(f"  Sint    : {len(ds_sint):,} segmentos")
    print(f"  Total   : {len(ds_train):,} segmentos")
    print(f"  Batches : {len(train_dl)}")



  CANDIDATO: CNN2D_logmel_2s
  Tipo: CNN2D | Filtro: log-mel | Dur: 2s
Val interna (calibracion threshold) : 10 pacientes
  Clase 0: 8
  Clase 1: 2
Train real (sin val)                : 53 pacientes
  Clase 0: 40
  Clase 1: 13
Sinteticos clase 1                  : 4,882 PNGs

  Class weights (segment level): C0=6,669, C1=7,319 → cw=[1.0487329959869385, 0.9555950164794922]

Dataset train aumentado:
  Real    : 9,106 segmentos
  Sint    : 4,882 segmentos
  Total   : 13,988 segmentos
  Batches : 110

  CANDIDATO: AST_gammatone_2s
  Tipo: AST | Filtro: gammatone | Dur: 2s
Val interna (calibracion threshold) : 10 pacientes
  Clase 0: 8
  Clase 1: 2
Train real (sin val)                : 53 pacientes
  Clase 0: 40
  Clase 1: 13
Sinteticos clase 1                  : 4,882 PNGs

  Class weights (segment level): C0=6,669, C1=7,319 → cw=[1.0487329959869385, 0.9555950164794922]

Dataset train aumentado:
  Real    : 9,106 segmentos
  Sint    : 4,882 segmentos
  Total   : 13,988 segmentos
  Batches

In [49]:
# ─── Celda de debug: Verificar dimensiones de entrada CNN ──────────────────
from torch.utils.data import DataLoader

# Probar con un stem real de log-mel
test_stems = [train_stems[0]]  # Primer stem de train
img_map_test = build_img_map(SPECS_ROOT / FILTER_NAME / f"{DURATION}s")

# Dataset CNN
ds_debug = StemDataset(test_stems, img_map_test, CNN_TF_VAL, mode="CNN")
dl_debug = DataLoader(ds_debug, batch_size=1, shuffle=False)

xb_debug, snames_debug = next(iter(dl_debug))
print(f"✅ DEBUG CNN input shape: {xb_debug.shape}")
print(f"   Esperado: (B, 3, H, W) → Ej: (1, 3, 128, 63)")
print(f"   Recibido: {xb_debug.shape}")

if xb_debug.shape[1] != 3:
    print(f"   ⚠️  ¡ALERTA! Canales en dim {xb_debug.shape[1]}, no en dim 3")

✅ DEBUG CNN input shape: torch.Size([1, 3, 128, 126])
   Esperado: (B, 3, H, W) → Ej: (1, 3, 128, 63)
   Recibido: torch.Size([1, 3, 128, 126])


In [50]:
# ─── Celda 7: Entrenamiento del modelo final por candidato ─────────────────

for cand in CANDIDATOS:
    # 1️ Variables por candidato (reemplaza las globales de Celda 1)
    cid         = cand["id"]
    MODEL_TYPE  = cand["model"]
    FILTER_NAME = cand["filter"]
    DURATION    = cand["dur"]
    IMG_H       = cand["img_h"]
    AST_H       = cand.get("ast_h", IMG_H)

    print(f"\n{'='*60}")
    print(f"  CANDIDATO: {cid}")
    print(f"  Tipo: {MODEL_TYPE} | Filtro: {FILTER_NAME} | Dur: {DURATION}s")
    print(f"{'='*60}")

    # 2️ Cargar split y sintéticos ESPECÍFICOS de este candidato
    json_path = SPLITS_DIR / f"split_{FILTER_NAME}_{DURATION}s.json"
    with open(json_path) as f: sp = json.load(f)
    train_stems  = sp["train"]["stems"]
    train_labels = np.array(sp["train"]["labels"], dtype=np.int64)
    test_stems   = sp["test"]["stems"]
    test_labels  = np.array(sp["test"]["labels"],  dtype=np.int64)

    # ✅ Ruta dinámica de sintéticos (por candidato)
    sintetico_dir = SINTETICOS_ROOT / FILTER_NAME / f"{DURATION}s" / "1_sintetico"
    sintetico_paths = sorted(sintetico_dir.glob("*.png")) if sintetico_dir.exists() else []

    # 3️ Split 85/15 para threshold (con datos REALES antes de augment)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=42)
    tr_sub_idx, val_idx = next(sss.split(train_stems, train_labels))
    val_stems       = [train_stems[i] for i in val_idx]
    val_labels_arr  = train_labels[val_idx]
    real_train_stems = [train_stems[i] for i in tr_sub_idx]
    real_train_lbl   = train_labels[tr_sub_idx]

    # ──────────────────────────────────────────────────────────────────────
    # TUS PRINTS ORIGINALES (INTACTOS)
    # ──────────────────────────────────────────────────────────────────────
    print(f"Val interna (calibracion threshold) : {len(val_stems)} pacientes")
    print(f"  Clase 0: {int((val_labels_arr==0).sum())}")
    print(f"  Clase 1: {int((val_labels_arr==1).sum())}")
    print(f"Train real (sin val)                : {len(real_train_stems)} pacientes")
    print(f"  Clase 0: {int((real_train_lbl==0).sum())}")
    print(f"  Clase 1: {int((real_train_lbl==1).sum())}")
    print(f"Sinteticos clase 1                  : {len(sintetico_paths):,} PNGs")

    # 4️ Construir img_map y target_w por candidato
    image_root = SPECS_ROOT / FILTER_NAME / f"{DURATION}s"
    img_map = {}
    for cls_str in ["0", "1"]:
        d = image_root / cls_str
        if not d.exists(): continue
        for p in sorted(d.glob("*.png")):
            stem = re.sub(r'_seg\d+$', '', p.stem)
            img_map.setdefault(stem, []).append(p)

    target_w = None
    if MODEL_TYPE == "AST":
        valid = [s for s in real_train_stems if s in img_map and len(img_map[s]) > 0]
        if valid: target_w = max(Image.open(img_map[s][0]).size[0] for s in valid)

    # 5️ Datasets real y sintético
    if MODEL_TYPE == "CNN2D":
        ds_real = RealDataset(real_train_stems, img_map, CNN_TF_TRAIN, mode="CNN")
        ds_sint = SinteticoDataset(sintetico_paths, CNN_TF_TRAIN, mode="CNN")
    else:
        ds_real = RealDataset(real_train_stems, img_map, AST_TF, mode="AST",
                              ast_h=AST_H, target_w=target_w)
        ds_sint = SinteticoDataset(sintetico_paths, AST_TF, mode="AST",
                                   ast_h=AST_H, target_w=target_w)

    # Concatenar real + sintético
    ds_train = ConcatDataset([ds_real, ds_sint])
    hp = HP[MODEL_TYPE]
    train_dl = DataLoader(ds_train, batch_size=hp["batch"], shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=False)

    # ──────────────────────────────────────────────────────────────────────
    # ✅ FIX CRÍTICO: class_weight a nivel de SEGMENTO (no paciente)
    # ──────────────────────────────────────────────────────────────────────
    c0_segs = sum(1 for _, l in ds_real if l == 0)
    c1_segs = sum(1 for _, l in ds_real if l == 1)
    c1_segs += len(ds_sint)  # Añadir sintéticos (todos clase 1)
    total = c0_segs + c1_segs
    cw_tensor = torch.tensor([total/(2*c0_segs), total/(2*c1_segs)], dtype=torch.float32).to(DEVICE)

    print(f"\n  Class weights (segment level): C0={c0_segs:,}, C1={c1_segs:,} → cw={cw_tensor.tolist()}")

    # ──────────────────────────────────────────────────────────────────────
    # TUS PRINTS ORIGINALES DE RESUMEN (INTACTOS)
    # ──────────────────────────────────────────────────────────────────────
    print(f"\nDataset train aumentado:")
    print(f"  Real    : {len(ds_real):,} segmentos")
    print(f"  Sint    : {len(ds_sint):,} segmentos")
    print(f"  Total   : {len(ds_train):,} segmentos")
    print(f"  Batches : {len(train_dl)}")

    # 6️ Entrenamiento (TU CÓDIGO ORIGINAL, solo indentado)
    model_ckpt_dir = RESULTS_ROOT / cid
    model_ckpt_dir.mkdir(parents=True, exist_ok=True)
    model_ckpt = model_ckpt_dir / "modelo_final.pt"

    if model_ckpt.exists():
        print("[CKPT] Modelo final ya entrenado. Cargando...")
        ckpt = torch.load(model_ckpt, map_location=DEVICE)
        if MODEL_TYPE == "CNN2D":
            model = CNN2D().to(DEVICE)
        else:
            model = ASTClassifier(AST_H, target_w, hp["n_unfreeze"]).to(DEVICE)
        model.load_state_dict(ckpt["model_state"])
        best_thr = ckpt["threshold"]
        print(f"  Epoch guardado : {ckpt['epoch']}")
        print(f"  Best loss      : {ckpt['best_loss']:.4f}")
        print(f"  Threshold      : {best_thr:.2f}")
    else:
        if MODEL_TYPE == "CNN2D":
            model     = CNN2D().to(DEVICE)
            optimizer = optim.Adam(model.parameters(), lr=hp["lr"])
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                            optimizer, 'min', factor=0.5, patience=3, min_lr=1e-7)
        else:
            model     = ASTClassifier(AST_H, target_w, hp["n_unfreeze"]).to(DEVICE)
            optimizer = optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=0.01)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                            optimizer, T_max=hp["epochs"], eta_min=1e-7)

        criterion = nn.BCELoss(reduction='none')
        best_loss, pat_cnt, best_state = float('inf'), 0, None
        t0 = time.time()

        print(f"Entrenando modelo final — {hp['epochs']} epocas max")
        print(f"  Train: {len(ds_train):,} segs (real + sintetico)")

        for epoch in range(hp["epochs"]):
            loss = train_epoch(model, train_dl, optimizer, criterion, cw_tensor)
            if MODEL_TYPE == "CNN2D":
                scheduler.step(loss)
            else:
                scheduler.step()
            if loss < best_loss:
                best_loss, pat_cnt = loss, 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                mark = "v"
            else:
                pat_cnt += 1
                mark = f"({pat_cnt}/{hp['patience']})"
            print(f"  Ep {epoch+1:02d}/{hp['epochs']}  "
                  f"loss {loss:.4f}  best {best_loss:.4f}  {mark}")
            if pat_cnt >= hp["patience"]:
                print("  Early stopping."); break

        model.load_state_dict(best_state)
        print(f"\nEntrenamiento: {(time.time()-t0)/60:.1f} min")

        # Calibrar threshold sobre val interna real
        print("Calibrando threshold sobre val interna...")
        val_probs = predict_stems(model, val_stems, img_map,
                                  hp["batch"], MODEL_TYPE,
                                  ast_h=AST_H if MODEL_TYPE=="AST" else None,
                                  target_w=target_w if MODEL_TYPE=="AST" else None)
        best_thr  = calibrate_thr(val_probs, val_stems, val_labels_arr)
        print(f"Threshold optimo: {best_thr:.2f}")

        # Guardar modelo (ruta por candidato)
        torch.save({
            "model_state" : best_state,
            "epoch"       : epoch + 1,
            "best_loss"   : best_loss,
            "threshold"   : best_thr,
            "cand"        : cand,  # Guardamos el candidato para auditar después
        }, model_ckpt)
        print(f"Modelo guardado: {model_ckpt}")

    # 🧹 Limpieza de memoria antes del siguiente candidato
    del model, ds_real, ds_sint, ds_train, train_dl, cw_tensor
    if 'optimizer' in locals(): del optimizer
    if 'scheduler' in locals(): del scheduler
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  ✅ {cid} completado. Memoria liberada.\n")


  CANDIDATO: CNN2D_logmel_2s
  Tipo: CNN2D | Filtro: log-mel | Dur: 2s
Val interna (calibracion threshold) : 10 pacientes
  Clase 0: 8
  Clase 1: 2
Train real (sin val)                : 53 pacientes
  Clase 0: 40
  Clase 1: 13
Sinteticos clase 1                  : 4,882 PNGs

  Class weights (segment level): C0=6,669, C1=7,319 → cw=[1.0487329959869385, 0.9555950164794922]

Dataset train aumentado:
  Real    : 9,106 segmentos
  Sint    : 4,882 segmentos
  Total   : 13,988 segmentos
  Batches : 110
Entrenando modelo final — 30 epocas max
  Train: 13,988 segs (real + sintetico)
  Ep 01/30  loss 0.4594  best 0.4594  v
  Ep 02/30  loss 0.3855  best 0.3855  v
  Ep 03/30  loss 0.3701  best 0.3701  v
  Ep 04/30  loss 0.3478  best 0.3478  v
  Ep 05/30  loss 0.3287  best 0.3287  v
  Ep 06/30  loss 0.3113  best 0.3113  v
  Ep 07/30  loss 0.2949  best 0.2949  v
  Ep 08/30  loss 0.2822  best 0.2822  v
  Ep 09/30  loss 0.2674  best 0.2674  v
  Ep 10/30  loss 0.2601  best 0.2601  v
  Ep 11/30  loss 

/tmp/ipykernel_88559/2260296601.py:100: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4480.)
  return t.T, stem
/tmp/ipykernel_88559/2260296601.py:100: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4480.)
  return t.T, stem
/tmp/ipykernel_88559/2260296601.py:100: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated an

Threshold optimo: 0.73
Modelo guardado: Ansiedad/fase3/resultado_final/CNN2D_logmel_2s/modelo_final.pt
  ✅ CNN2D_logmel_2s completado. Memoria liberada.


  CANDIDATO: AST_gammatone_2s
  Tipo: AST | Filtro: gammatone | Dur: 2s
Val interna (calibracion threshold) : 10 pacientes
  Clase 0: 8
  Clase 1: 2
Train real (sin val)                : 53 pacientes
  Clase 0: 40
  Clase 1: 13
Sinteticos clase 1                  : 4,882 PNGs

  Class weights (segment level): C0=6,669, C1=7,319 → cw=[1.0487329959869385, 0.9555950164794922]

Dataset train aumentado:
  Real    : 9,106 segmentos
  Sint    : 4,882 segmentos
  Total   : 13,988 segmentos
  Batches : 438


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17785.73it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  AST entrenables: 21,462,273
Entrenando modelo final — 20 epocas max
  Train: 13,988 segs (real + sintetico)
  Ep 01/20  loss 0.3717  best 0.3717  v
  Ep 02/20  loss 0.3155  best 0.3155  v
  Ep 03/20  loss 0.2738  best 0.2738  v
  Ep 04/20  loss 0.2265  best 0.2265  v
  Ep 05/20  loss 0.1867  best 0.1867  v
  Ep 06/20  loss 0.1358  best 0.1358  v
  Ep 07/20  loss 0.1046  best 0.1046  v
  Ep 08/20  loss 0.0764  best 0.0764  v
  Ep 09/20  loss 0.0430  best 0.0430  v
  Ep 10/20  loss 0.0230  best 0.0230  v
  Ep 11/20  loss 0.0147  best 0.0147  v
  Ep 12/20  loss 0.0076  best 0.0076  v
  Ep 13/20  loss 0.0031  best 0.0031  v
  Ep 14/20  loss 0.0002  best 0.0002  v
  Ep 15/20  loss 0.0000  best 0.0000  v
  Ep 16/20  loss 0.0000  best 0.0000  v
  Ep 17/20  loss 0.0000  best 0.0000  v
  Ep 18/20  loss 0.0000  best 0.0000  v
  Ep 19/20  loss 0.0000  best 0.0000  v
  Ep 20/20  loss 0.0000  best 0.0000  v

Entrenamiento: 2.7 min
Calibrando threshold sobre val interna...
Threshold optimo: 0.15
M

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11120.14it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  AST entrenables: 21,462,273
Entrenando modelo final — 20 epocas max
  Train: 9,290 segs (real + sintetico)
  Ep 01/20  loss 0.3782  best 0.3782  v
  Ep 02/20  loss 0.3243  best 0.3243  v
  Ep 03/20  loss 0.2755  best 0.2755  v
  Ep 04/20  loss 0.2268  best 0.2268  v
  Ep 05/20  loss 0.1775  best 0.1775  v
  Ep 06/20  loss 0.1318  best 0.1318  v
  Ep 07/20  loss 0.0833  best 0.0833  v
  Ep 08/20  loss 0.0569  best 0.0569  v
  Ep 09/20  loss 0.0359  best 0.0359  v
  Ep 10/20  loss 0.0134  best 0.0134  v
  Ep 11/20  loss 0.0068  best 0.0068  v
  Ep 12/20  loss 0.0012  best 0.0012  v
  Ep 13/20  loss 0.0005  best 0.0005  v
  Ep 14/20  loss 0.0001  best 0.0001  v
  Ep 15/20  loss 0.0001  best 0.0001  v
  Ep 16/20  loss 0.0000  best 0.0000  v
  Ep 17/20  loss 0.0000  best 0.0000  v
  Ep 18/20  loss 0.0000  best 0.0000  v
  Ep 19/20  loss 0.0000  best 0.0000  v
  Ep 20/20  loss 0.0000  best 0.0000  v

Entrenamiento: 2.6 min
Calibrando threshold sobre val interna...
Threshold optimo: 0.13
Mo

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20962.57it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  AST entrenables: 21,462,273
Entrenando modelo final — 20 epocas max
  Train: 6,941 segs (real + sintetico)
  Ep 01/20  loss 0.3762  best 0.3762  v
  Ep 02/20  loss 0.3126  best 0.3126  v
  Ep 03/20  loss 0.2632  best 0.2632  v
  Ep 04/20  loss 0.2021  best 0.2021  v
  Ep 05/20  loss 0.1530  best 0.1530  v
  Ep 06/20  loss 0.1031  best 0.1031  v
  Ep 07/20  loss 0.0656  best 0.0656  v
  Ep 08/20  loss 0.0303  best 0.0303  v
  Ep 09/20  loss 0.0135  best 0.0135  v
  Ep 10/20  loss 0.0051  best 0.0051  v
  Ep 11/20  loss 0.0008  best 0.0008  v
  Ep 12/20  loss 0.0006  best 0.0006  v
  Ep 13/20  loss 0.0002  best 0.0002  v
  Ep 14/20  loss 0.0001  best 0.0001  v
  Ep 15/20  loss 0.0001  best 0.0001  v
  Ep 16/20  loss 0.0001  best 0.0001  v
  Ep 17/20  loss 0.0001  best 0.0001  v
  Ep 18/20  loss 0.0001  best 0.0001  v
  Ep 19/20  loss 0.0001  best 0.0001  v
  Ep 20/20  loss 0.0001  best 0.0001  (1/4)

Entrenamiento: 2.9 min
Calibrando threshold sobre val interna...
Threshold optimo: 0.1

In [53]:
# ─── Celda 8: EVALUACIÓN FINAL EN TEST (solo carga y predice) ─────────────

print("\n" + "="*72)
print("EVALUACIÓN FINAL EN TEST (CARGANDO MODELOS ENTRENADOS)")
print("="*72)

resultados_finales = {}

for cand in CANDIDATOS:
    cid = cand["id"]
    MODEL_TYPE = cand["model"]
    FILTER_NAME = cand["filter"]
    DURATION = cand["dur"]
    IMG_H = cand["img_h"]
    AST_H = cand.get("ast_h", IMG_H)
    hp = HP[MODEL_TYPE]

    print(f"\n{'='*60}")
    print(f"  EVALUANDO: {cid}")
    print(f"{'='*60}")

    # 1️⃣ Cargar split de test
    json_path = SPLITS_DIR / f"split_{FILTER_NAME}_{DURATION}s.json"
    with open(json_path) as f: sp = json.load(f)
    test_stems  = sp["test"]["stems"]
    test_labels = np.array(sp["test"]["labels"], dtype=np.int64)

    # 2️⃣ Construir img_map solo con stems de test
    image_root = SPECS_ROOT / FILTER_NAME / f"{DURATION}s"
    img_map = {}
    for cls_str in ["0", "1"]:
        d = image_root / cls_str
        if not d.exists(): continue
        for p in sorted(d.glob("*.png")):
            stem = re.sub(r'_seg\d+$', '', p.stem)
            img_map.setdefault(stem, []).append(p)

    # 3️⃣ Cargar checkpoint entrenado en Celda 7
    model_ckpt = RESULTS_ROOT / cid / "modelo_final.pt"
    if not model_ckpt.exists():
        print(f"  [SKIP] No hay modelo guardado para {cid}. Ejecuta Celda 7 primero.")
        continue

    ckpt = torch.load(model_ckpt, map_location=DEVICE)
    best_thr = ckpt["threshold"]
    print(f"  Modelo cargado | Threshold: {best_thr:.2f}")

    # 4️⃣ Instanciar modelo y cargar pesos
    if MODEL_TYPE == "CNN2D":
        model = CNN2D().to(DEVICE)
    else:
        # Recalcular target_w solo para test (consistente con AST)
        valid_stems = [s for s in test_stems if s in img_map and len(img_map[s]) > 0]
        target_w = max(Image.open(img_map[s][0]).size[0] for s in valid_stems) if valid_stems else 128
        model = ASTClassifier(AST_H, target_w, hp["n_unfreeze"]).to(DEVICE)

    model.load_state_dict(ckpt["model_state"])
    model.eval()
    print(f"  Test: {len(test_stems)} pacientes | AST target_w: {target_w if MODEL_TYPE=='AST' else 'N/A'}px")

    # 5️⃣ Predicción en test
    test_probs = predict_stems(model, test_stems, img_map, hp["batch"], MODEL_TYPE,
                               ast_h=AST_H if MODEL_TYPE=="AST" else None,
                               target_w=target_w if MODEL_TYPE=="AST" else None)

    # 6️⃣ Calcular métricas
    true, pred, prob = [], [], []
    for stem, lbl in zip(test_stems, test_labels):
        p = test_probs[stem]
        true.append(int(lbl))
        pred.append(1 if p > best_thr else 0)
        prob.append(p)

    true_arr, prob_arr = np.array(true), np.array(prob)
    fpr, tpr, _ = roc_curve(true_arr, prob_arr)
    roc_auc = float(auc(fpr, tpr))

    metricas_test = {
        "accuracy" : float(accuracy_score(true, pred)),
        "precision": float(precision_score(true, pred, zero_division=0)),
        "recall"   : float(recall_score(true, pred, zero_division=0)),
        "f1"       : float(f1_score(true, pred, zero_division=0)),
        "f1_macro" : float(f1_score(true, pred, average='macro', zero_division=0)),
        "auc"      : roc_auc,
        "threshold": best_thr,
    }

    print(f"  Accuracy  : {metricas_test['accuracy']:.4f}")
    print(f"  F1-Macro  : {metricas_test['f1_macro']:.4f}")
    print(f"  AUC-ROC   : {metricas_test['auc']:.4f}")

    # 7️⃣ Comparación vs mini CV
    mini_cv_path = Path("Ansiedad/fase3/mini_cv/mini_cv_checkpoint.json")
    if mini_cv_path.exists():
        mini_cv_data = json.load(open(mini_cv_path))
        cv_folds = mini_cv_data.get(cid, [])
        if cv_folds:
            cv_f1mac = float(np.mean([m["f1mac"] for m in cv_folds]))
            cv_auc   = float(np.mean([m["auc"]   for m in cv_folds]))
        else: cv_f1mac, cv_auc = 0.0, 0.0
    else: cv_f1mac, cv_auc = 0.0, 0.0

    delta_f1  = metricas_test['f1_macro'] - cv_f1mac
    delta_auc = metricas_test['auc']      - cv_auc
    print(f"  ΔF1-Macro vs CV: {delta_f1:+.3f} | ΔAUC vs CV: {delta_auc:+.3f}")

    # 8️⃣ Guardar resultados (incluyendo datos crudos para Celda 9)
    resultado_final = {
        "modelo": cid, "filter": FILTER_NAME, "duration": DURATION,
        "threshold": best_thr, "n_test_pac": len(test_stems),
        "metricas_test": metricas_test,
        "mini_cv_f1mac": cv_f1mac, "mini_cv_auc": cv_auc,
        "delta_f1mac": delta_f1, "delta_auc": delta_auc,
        "true_test": true, "pred_test": pred, "prob_test": prob,
        "roc_fpr": fpr.tolist(), "roc_tpr": tpr.tolist()
    }
    with open(RESULTS_ROOT / cid / "resultado_final.json", "w") as f:
        json.dump(resultado_final, f, indent=2)

    resultados_finales[cid] = resultado_final
    del model; gc.collect(); torch.cuda.empty_cache()

print(f"\n{'='*72}")
print("EVALUACIÓN FINAL COMPLETADA")
print(f"{'='*72}")
for cid, res in resultados_finales.items():
    print(f"  {cid:25s} → F1-Macro: {res['metricas_test']['f1_macro']:.3f} | AUC: {res['metricas_test']['auc']:.3f}")


EVALUACIÓN FINAL EN TEST (CARGANDO MODELOS ENTRENADOS)

  EVALUANDO: CNN2D_logmel_2s
  Modelo cargado | Threshold: 0.27
  Test: 16 pacientes | AST target_w: N/Apx
  Accuracy  : 0.7500
  F1-Macro  : 0.4286
  AUC-ROC   : 0.7917
  ΔF1-Macro vs CV: -0.000 | ΔAUC vs CV: +0.350

  EVALUANDO: AST_gammatone_2s
  Modelo cargado | Threshold: 0.15


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19875.85it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  AST entrenables: 21,462,273
  Test: 16 pacientes | AST target_w: 63px
  Accuracy  : 0.5625
  F1-Macro  : 0.5466
  AUC-ROC   : 0.7708
  ΔF1-Macro vs CV: +0.021 | ΔAUC vs CV: +0.152

  EVALUANDO: AST_gammatone_3s
  Modelo cargado | Threshold: 0.13


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18320.56it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  AST entrenables: 21,462,273
  Test: 16 pacientes | AST target_w: 94px
  Accuracy  : 0.5000
  F1-Macro  : 0.4921
  AUC-ROC   : 0.7292
  ΔF1-Macro vs CV: -0.028 | ΔAUC vs CV: +0.119

  EVALUANDO: AST_gammatone_4s
  Modelo cargado | Threshold: 0.15


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22253.62it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  AST entrenables: 21,462,273
  Test: 16 pacientes | AST target_w: 126px
  Accuracy  : 0.6875
  F1-Macro  : 0.6537
  AUC-ROC   : 0.6875
  ΔF1-Macro vs CV: +0.156 | ΔAUC vs CV: +0.106

EVALUACIÓN FINAL COMPLETADA
  CNN2D_logmel_2s           → F1-Macro: 0.429 | AUC: 0.792
  AST_gammatone_2s          → F1-Macro: 0.547 | AUC: 0.771
  AST_gammatone_3s          → F1-Macro: 0.492 | AUC: 0.729
  AST_gammatone_4s          → F1-Macro: 0.654 | AUC: 0.688


In [54]:
# ─── Celda 9: Figuras finales por candidato + resumen global ───────────────

print("\n" + "="*72)
print("GENERANDO FIGURAS FINALES")
print("="*72)

# Lista para acumular resultados y hacer la comparación global al final
resumen_comparativo = []

for cand in CANDIDATOS:
    cid = cand["id"]
    print(f"\n{'='*60}")
    print(f"  GENERANDO GRÁFICOS PARA: {cid}")
    print(f"{'='*60}")

    # 1️⃣ Cargar datos crudos del JSON (que guardamos en Celda 8)
    resultado_path = RESULTS_ROOT / cid / "resultado_final.json"
    if not resultado_path.exists():
        print(f"  [SKIP] No hay resultados para {cid}. Salta a la siguiente Celda.")
        continue

    with open(resultado_path) as f:
        res = json.load(f)

    metricas_test = res["metricas_test"]

    # Recuperar datos crudos
    true      = res["true_test"]
    pred      = res["pred_test"]
    prob      = res["prob_test"]
    fpr       = np.array(res["roc_fpr"])
    tpr       = np.array(res["roc_tpr"])
    roc_auc   = metricas_test["auc"]

    # Directorio de salida para este candidato
    out_dir = RESULTS_ROOT / cid
    out_dir.mkdir(exist_ok=True)

    # ──────────────────────────────────────────────────────────────────────
    # Figura 1: Confusion Matrix
    # ──────────────────────────────────────────────────────────────────────
    cm = confusion_matrix(true, pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=[f"Sin {CONDITION}", f"Con {CONDITION}"],
                yticklabels=[f"Sin {CONDITION}", f"Con {CONDITION}"])
    ax.set_title(f"Confusion Matrix — Test\n{cid} + VAE augmentation")
    ax.set_ylabel("Real"); ax.set_xlabel("Predicción")
    plt.tight_layout()
    plt.savefig(out_dir / "confusion_matrix.png", dpi=150)
    plt.close()
    print(f"  ✅ Confusion Matrix guardada")

    # ──────────────────────────────────────────────────────────────────────
    # Figura 2: ROC Curve
    # ──────────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, lw=2.5, color='navy', label=f"AUC = {roc_auc:.3f}")
    ax.fill_between(fpr, tpr, alpha=0.08, color='navy')
    ax.plot([0,1],[0,1],'k--', lw=1)
    ax.axvline(x=0, color='gray', linewidth=0.5)
    ax.set_xlabel("Tasa de Falsos Positivos")
    ax.set_ylabel("Tasa de Verdaderos Positivos")
    ax.set_title(f"ROC — Test\n{cid} + VAE augmentation")
    ax.legend(loc='lower right', fontsize=11)
    plt.tight_layout()
    plt.savefig(out_dir / "roc_curve.png", dpi=150)
    plt.close()
    print(f"  ✅ ROC Curve guardada")

    # ──────────────────────────────────────────────────────────────────────
    # Figura 3: Barras Comparativas Mini CV vs Final
    # ──────────────────────────────────────────────────────────────────────
    metricas_cmp = ["f1_macro", "auc"]
    labels_cmp   = ["F1-Macro", "AUC-ROC"]
    val_cv   = [res["mini_cv_f1mac"], res["mini_cv_auc"]]
    val_test = [metricas_test["f1_macro"], metricas_test["auc"]]

    x     = np.arange(len(metricas_cmp))
    width = 0.3
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(x - width/2, val_cv,   width, label="Mini CV (sin augment)", color="steelblue",  alpha=0.8)
    ax.bar(x + width/2, val_test, width, label="Test final (con VAE)",  color="darkorange", alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels(labels_cmp, fontsize=12)
    ax.set_ylim(0, 1.05); ax.set_ylabel("Valor")
    ax.set_title(f"Mini CV vs Resultado Final\n{cid}")
    ax.legend(fontsize=10)
    for i, (v1, v2) in enumerate(zip(val_cv, val_test)):
        ax.text(i - width/2, v1 + 0.02, f"{v1:.3f}", ha='center', fontsize=9)
        ax.text(i + width/2, v2 + 0.02, f"{v2:.3f}", ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(out_dir / "comparacion_cv_vs_test.png", dpi=150)
    plt.close()
    print(f"  ✅ Comparativa guardada")

    resumen_comparativo.append(res)
    gc.collect()

# ──────────────────────────────────────────────────────────────────────────
# FIGURA FINAL: Resumen Global (Todos los candidatos)
# ──────────────────────────────────────────────────────────────────────────
if len(resumen_comparativo) > 1:
    print(f"\n{'='*60}")
    print("GENERANDO RESUMEN GLOBAL DE TODOS LOS CANDIDATOS")
    print(f"{'='*60}")

    ids = [r["modelo"] for r in resumen_comparativo]
    f1_test = [r["metricas_test"]["f1_macro"] for r in resumen_comparativo]
    auc_test = [r["metricas_test"]["auc"] for r in resumen_comparativo]
    f1_cv   = [r["mini_cv_f1mac"] for r in resumen_comparativo]
    auc_cv  = [r["mini_cv_auc"] for r in resumen_comparativo]

    x = np.arange(len(ids))
    width = 0.35

    # F1-Macro: CV vs Test
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.bar(x - width/2, f1_cv, width, label="Mini CV", color="steelblue", alpha=0.8)
    ax1.bar(x + width/2, f1_test, width, label="Test Final", color="darkorange", alpha=0.8)
    ax1.set_xticks(x)
    ax1.set_xticklabels([c.replace("CNN2D_","CNN ").replace("AST_","AST ") for c in ids], rotation=15, ha='right')
    ax1.set_ylabel("F1-Macro"); ax1.set_ylim(0, 1.05); ax1.legend(); ax1.grid(axis='y', alpha=0.3)
    ax1.set_title("F1-Macro: CV vs Test (Todos los candidatos)")

    # AUC-ROC: CV vs Test
    ax2.bar(x - width/2, auc_cv, width, label="Mini CV", color="steelblue", alpha=0.8)
    ax2.bar(x + width/2, auc_test, width, label="Test Final", color="darkorange", alpha=0.8)
    ax2.set_xticks(x)
    ax2.set_xticklabels([c.replace("CNN2D_","CNN ").replace("AST_","AST ") for c in ids], rotation=15, ha='right')
    ax2.set_ylabel("AUC-ROC"); ax2.set_ylim(0, 1.05); ax2.legend(); ax2.grid(axis='y', alpha=0.3)
    ax2.set_title("AUC-ROC: CV vs Test (Todos los candidatos)")

    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / "resumen_global_comparacion.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ Resumen Global guardado en: {RESULTS_ROOT / 'resumen_global_comparacion.png'}")

print(f"\n{'='*72}")
print("FASE 3 COMPLETADA")
print(f"Resultados finales en: {RESULTS_ROOT}")
print("="*72)


GENERANDO FIGURAS FINALES

  GENERANDO GRÁFICOS PARA: CNN2D_logmel_2s
  ✅ Confusion Matrix guardada
  ✅ ROC Curve guardada
  ✅ Comparativa guardada

  GENERANDO GRÁFICOS PARA: AST_gammatone_2s
  ✅ Confusion Matrix guardada
  ✅ ROC Curve guardada
  ✅ Comparativa guardada

  GENERANDO GRÁFICOS PARA: AST_gammatone_3s
  ✅ Confusion Matrix guardada
  ✅ ROC Curve guardada
  ✅ Comparativa guardada

  GENERANDO GRÁFICOS PARA: AST_gammatone_4s
  ✅ Confusion Matrix guardada
  ✅ ROC Curve guardada
  ✅ Comparativa guardada

GENERANDO RESUMEN GLOBAL DE TODOS LOS CANDIDATOS
  ✅ Resumen Global guardado en: Ansiedad/fase3/resultado_final/resumen_global_comparacion.png

FASE 3 COMPLETADA
Resultados finales en: Ansiedad/fase3/resultado_final
